# Agent Harness Engineering：把模型调用变成可控的 Agent 运行时

**三秒结论：Agent Harness 是模型外部的运行和控制系统。** 它组织上下文、调用模型、执行工具、控制权限、保存状态、处理失败并验证结果。

一个只会循环调用模型和工具的程序可以完成演示。一个需要修改文件、访问业务系统、运行数分钟、支持中断恢复并接受审计的 Agent，需要 Harness。

## 1 结论与学习目标

学完本章，你需要理解以下内容：

1. Harness 的定义、产生背景和职责边界。
2. Harness 与模型、Agent Loop、Framework、Skill、CLI、MCP、Memory、Guardrails 的关系。
3. 一次任务如何经过上下文装配、模型决策、权限检查、工具执行、状态保存和结果验收。
4. 如何处理大输出、超时、预算、取消、崩溃窗口、幂等和 Checkpoint 恢复。
5. 哪些场景只需直接调用模型，哪些场景需要完整 Harness。

本章使用“跨文件 Python API 迁移”作为贯穿案例。代码用于观察运行时协议；理解 Harness 不要求先读懂完整框架代码。

### 1.1 七条生产不变量

先记住七条约束，后文所有组件都从这些约束展开：

1. 每个动作先经过策略判断，再进入执行环境。
2. 工具结果持久化后，Harness 才开始下一轮模型调用。
3. 大结果进入 Artifact Store，上下文只保留摘要和引用。
4. 有副作用的动作携带稳定 `action_id` 或业务幂等键。
5. 模型声明完成后，Verifier 独立执行验收。
6. 每次运行都有轮数、Token、时间、费用和工具调用预算。
7. 所有事件都能关联到 `thread_id`、`run_id`、`turn_id` 和 `action_id`。

## 2 What：Harness 本质上是什么

### 2.1 定义

**Agent Harness 是承载 Agent 的运行时和控制层。** 模型负责根据输入生成下一步内容，Harness 负责把这一步放进真实系统中安全、持续、可恢复地执行。

可以用一个工程公式表示：

```text
Agent System
  = Model Policy
  + Harness Control Plane
  + Tools / Execution Environment
  + State / Artifact Store
```

- `Model Policy`：模型根据当前上下文选择下一步。
- `Harness Control Plane`：调度、授权、预算、恢复和验收。
- `Tools / Execution Environment`：文件、CLI、MCP、浏览器、数据库和沙箱。
- `State / Artifact Store`：事件、Checkpoint、工具结果和任务产物。

### 2.2 最小结构

```mermaid
flowchart TD
    U["用户或上层应用"] --> H["Harness 控制层"]
    H --> C["Context Builder"]
    C --> M["大模型"]
    M --> H
    H --> P["Policy 与审批"]
    P --> R["Tool Runtime"]
    R --> T["CLI / MCP / 本地工具"]
    H <--> S["Event / Checkpoint Store"]
    H <--> A["Artifact Store"]
    H --> V["Verifier"]
    V --> H
    H --> U
```

图中的箭头代表控制流和数据流。模型不会直接修改文件；它产生动作请求，Harness 完成检查、执行和记录。

### 2.3 职责边界

| 问题 | 主要负责人 |
| --- | --- |
| 下一步应该做什么 | 模型 |
| 本轮给模型哪些信息 | Context Builder |
| 动作是否允许 | Policy Engine |
| 工具在哪里执行 | Tool Runtime / Sandbox |
| 任务执行到哪里 | Event Store / Checkpoint |
| 大型结果放在哪里 | Artifact Store |
| 失败后继续还是停止 | Supervisor |
| 结果是否满足要求 | Verifier |

`Harness` 没有单一标准类名。产品可能使用 Runtime、Agent Server、Orchestrator、Runner 或 App Server 表达相近职责。判断依据是系统承担的职责，不依赖命名。

## 3 Why：Harness 为什么会产生

### 3.1 裸 Agent Loop 的能力边界

最小循环只有四步：

```text
调用模型 → 获得 Tool Call → 执行工具 → 回传 Tool Result
     ↑                                      ↓
     └──────────────── 下一轮 ──────────────┘
```

这个循环解释了 Agent 如何行动。它没有回答权限、持久化、恢复、预算和验收问题。任务从“查天气”升级到“修改代码并提交结果”后，这些问题会同时出现。

### 3.2 从模型能力推导 Harness 组件

| 已知事实 | 直接风险 | Harness 对策 |
| --- | --- | --- |
| 模型输出具有概率性 | 参数错误、动作漂移 | Schema、工具白名单、Verifier |
| 模型只能生成内容 | 无法直接操作环境 | Tool Runtime、CLI、MCP |
| 上下文窗口有限 | 历史和工具输出挤占空间 | Context Builder、Compaction、Artifact Store |
| 工具可能产生副作用 | 重复写入、越权访问 | Policy、Sandbox、Approval、Idempotency |
| 网络和进程会失败 | 任务停在中间 | Timeout、Retry、Checkpoint、Resume |
| 客户端可能断开 | 运行状态丢失 | Thread、Event Store、后台 Worker |
| 模型可能提前宣布完成 | 结果未通过测试 | 独立 Verifier、Acceptance Gate |
| 多个用户共享服务 | 数据、凭据和状态串用 | Identity、Tenant Isolation、Audit |

Harness 的组件来自具体故障，不需要先设计一个大平台。生产系统通常从最小循环开始，随着副作用、持续时间和并发量增加，逐步补齐控制能力。

### 3.3 行业背景：模型能力提高后，运行环境成为主要变量

Agent 早期实践主要优化 Prompt 和工具选择。代码任务逐渐变长后，模型需要理解仓库规则、持续调用工具、处理测试反馈、跨客户端恢复任务，并在受限环境内执行命令。

OpenAI 对 Codex agent loop 的拆解包含初始 Prompt 构造、工具循环、不断增长的上下文、Prompt Cache 和 Compaction；Codex App Server 进一步提供 Thread 生命周期、持久化、认证、沙箱、MCP、Skill 和双向审批事件。[1][2] OpenAI 的 Harness Engineering 实践强调仓库知识、日志、指标和测试必须采用 Agent 可读取的形式。[4]

Anthropic 建议长任务使用持久化产物跨越上下文边界，并随着模型能力变化持续检验 Harness 假设。[6][7] 其生产架构进一步拆成 Brain、Hands 和 Session：模型与 Harness 负责决策，沙箱负责执行，事件日志负责持续状态。[8] “Building effective agents”则建议优先采用简单、可组合的模式，再按任务需要增加自主性。[5]

SWE-agent 的研究表明，Agent-Computer Interface 的设计会直接影响模型完成软件工程任务的能力。[9]

这些系统的共同方向是：模型能力与运行环境共同决定 Agent 的可用性。

## 4 Integration：Harness 与其他知识的关系

### 4.1 关系图

```mermaid
flowchart TD
    F["Agent Framework"] --> H["Harness"]
    H --> L["Agent Loop"]
    H --> C["Context / Memory"]
    H --> P["Policy / Guardrails"]
    H --> E["Event / Checkpoint"]
    H --> R["Tool Runtime"]
    R --> FC["Function Calling"]
    R --> CLI["Agent-facing CLI"]
    R --> MCP["MCP"]
    H --> SK["Skill 加载"]
    H --> OT["Evals / Tracing"]
```

### 4.2 各技术在体系中的位置

| 技术 | 解决的问题 | 与 Harness 的搭配方式 |
| --- | --- | --- |
| Agent Loop | 模型、动作和观察如何循环 | Harness 内部的核心控制循环 |
| Function Calling | 模型如何表达结构化动作 | Harness 解析动作并交给 Tool Runtime |
| CLI | Agent 如何稳定调用业务能力 | 作为 Tool Runtime 的一种工具接口 |
| MCP | 客户端如何发现并调用外部能力 | 作为工具和资源的标准连接协议 |
| Skill | 模型如何获得任务方法和资源索引 | Context Builder 按需加载 Skill |
| Context / Memory | 本轮信息和长期信息如何组织 | 为 Context Builder 提供材料 |
| Framework | 如何减少状态机和集成代码 | 提供 Harness 的实现脚手架 |
| Guardrails | 输入、动作和输出如何受约束 | 嵌入模型调用前、工具执行前和输出前 |
| Evals / Tracing | 如何评估和观察系统 | 消费 Harness 产生的事件和结果 |

第十一章会使用统一场景比较框架。第十二章会深入 Evals、Tracing、Guardrails 和人审。本章只实现它们与运行时直接相连的最小接口。

## 5 How：Harness 如何运行一次任务

### 5.1 三组参与者

生产运行时可以按职责拆成三组：[8]

| 分组 | 包含内容 | 失败后的处理 |
| --- | --- | --- |
| Brain | 模型适配器、Context Builder、Agent Loop、Policy、Supervisor | 可重启并从事件恢复 |
| Hands | 沙箱、工作目录、CLI、MCP、浏览器和本地工具 | 可销毁、重建或替换 |
| Session | Thread、Turn、Event、Checkpoint、Artifact | 持久化并支持重连 |

这种拆分降低故障耦合。模型请求失败不应删除工作目录，沙箱重建不应丢失任务历史，客户端断开不应终止后台任务。

### 5.2 Thread、Run、Turn、Item 与 Event

| 对象 | 含义 | 示例 |
| --- | --- | --- |
| Thread | 一段可持续、可恢复的任务上下文 | 一次代码迁移任务 |
| Run | Thread 中的一次执行尝试 | 首次执行、失败后重试 |
| Turn | 一次用户输入到 Agent 响应的处理周期 | “继续完成迁移” |
| Item | Turn 内的语义对象 | 模型消息、Tool Call、Tool Result |
| Event | 运行时发生的不可变事实 | `tool.started`、`tool.completed` |

业务表保存当前状态，事件日志保存发生过程。只保存最终 `status=failed` 无法回答“哪个工具失败、是否产生副作用、恢复时从哪里继续”。

In [1]:
from dataclasses import asdict, dataclass, field
from datetime import datetime, UTC
from typing import Any
import json
import uuid

@dataclass(frozen=True)
class Event:
    # seq 决定同一运行中的事件顺序；事件写入后不再修改。
    seq: int
    event_type: str
    thread_id: str
    run_id: str
    turn_id: str
    action_id: str | None = None
    payload: dict[str, Any] = field(default_factory=dict)
    created_at: str = field(default_factory=lambda: datetime.now(UTC).isoformat())

sample_event = Event(
    seq=4,
    event_type="tool.completed",
    thread_id="thread-42",
    run_id="run-2",
    turn_id="turn-7",
    action_id="action-read-pricing",
    payload={"tool": "read_file", "artifact_ref": "artifact://sha256/abc"},
)
print(json.dumps(asdict(sample_event), ensure_ascii=False, indent=2))
assert sample_event.action_id == "action-read-pricing"

{
  "seq": 4,
  "event_type": "tool.completed",
  "thread_id": "thread-42",
  "run_id": "run-2",
  "turn_id": "turn-7",
  "action_id": "action-read-pricing",
  "payload": {
    "tool": "read_file",
    "artifact_ref": "artifact://sha256/abc"
  },
  "created_at": "2026-07-27T13:07:24.062865+00:00"
}


### 5.3 完整时序

```mermaid
sequenceDiagram
    participant U as 用户
    participant H as Harness
    participant S as State Store
    participant M as 模型
    participant P as Policy
    participant R as Tool Runtime
    participant V as Verifier

    U->>H: 提交任务
    H->>S: 创建 Thread 与 Run
    H->>H: 构造本轮上下文
    H->>M: messages + tools
    M-->>H: Tool Call
    H->>P: 检查动作和参数
    P-->>H: ALLOW / DENY / REQUIRE_APPROVAL
    H->>R: 执行动作
    R-->>H: Tool Result
    H->>S: 持久化事件和结果
    H->>M: Tool Result 或 Artifact 引用
    M-->>H: 声明任务完成
    H->>V: 执行测试和验收
    V-->>H: PASS / FAIL
    H->>S: 保存最终状态
    H-->>U: 返回已验证结果
```

顺序中最重要的一点是：`Tool Result` 先持久化，再进入下一轮。否则模型已经基于结果继续推理，系统恢复后却找不到这份结果。

### 5.4 运行状态机

```mermaid
stateDiagram-v2
    [*] --> CREATED
    CREATED --> RUNNING
    RUNNING --> WAITING_APPROVAL
    WAITING_APPROVAL --> RUNNING: approved
    WAITING_APPROVAL --> FAILED: denied
    RUNNING --> PAUSED: checkpoint
    PAUSED --> RUNNING: resume
    RUNNING --> VERIFYING: model_done
    VERIFYING --> RUNNING: verification_failed
    VERIFYING --> SUCCEEDED: verification_passed
    RUNNING --> CANCELLED: cancel
    RUNNING --> FAILED: unrecoverable_error
    SUCCEEDED --> [*]
    CANCELLED --> [*]
    FAILED --> [*]
```

模型文本“完成”只触发 `VERIFYING`。只有验收通过才能进入 `SUCCEEDED`。

### 5.5 Context Builder：决定模型这一轮看到什么

Context Builder 每轮重新装配输入，常见顺序如下：

1. 稳定系统指令和安全规则。
2. 当前工具 Schema。
3. 仓库级规则与按需加载的 Skill。
4. 当前任务和验收标准。
5. 运行状态摘要、未完成计划和最近观察。
6. Artifact 引用及其读取方法。

稳定内容放在前缀，有利于提供方的 Prompt Cache 命中；动态事件放在后部。[1] Cache 由模型服务提供，Harness 负责保持可复用前缀稳定，并记录缓存相关指标。

## 6 渐进实验：跨文件价格 API 迁移

### 6.1 案例与验收标准

示例项目使用旧函数 `legacy_price(amount)` 计算九折价格。任务要求引入 `PricePolicy.apply(amount)`，迁移生产代码和测试。

验收标准只有两条：

1. 全部单元测试通过。
2. 所有 Python 文件中不再出现 `legacy_price`。

这个案例包含跨文件读取、写入、搜索、测试和结果验证，规模足以暴露 Harness 问题，同时可以在 Notebook 内完整运行。

In [2]:
from pathlib import Path
import subprocess
import sys
import tempfile

LAB_ROOT = Path(tempfile.mkdtemp(prefix="harness_ch10_"))

PROJECT_FILES = {
    "pricing.py": '''def legacy_price(amount: float) -> float:\n    return round(amount * 0.9, 2)\n''',
    "checkout.py": '''from pricing import legacy_price\n\ndef checkout_total(amount: float) -> float:\n    return legacy_price(amount)\n''',
    "report.py": '''from pricing import legacy_price\n\ndef display_price(amount: float) -> str:\n    return f"¥{legacy_price(amount):.2f}"\n''',
    "test_pricing.py": '''import unittest\nfrom pricing import legacy_price\nfrom checkout import checkout_total\nfrom report import display_price\n\nclass PricingTest(unittest.TestCase):\n    def test_discount(self):\n        self.assertEqual(legacy_price(100), 90.0)\n\n    def test_checkout(self):\n        self.assertEqual(checkout_total(200), 180.0)\n\n    def test_report(self):\n        self.assertEqual(display_price(50), "¥45.00")\n\nif __name__ == "__main__":\n    unittest.main()\n''',
}

def make_workspace(name: str) -> Path:
    # 创建一份互不干扰的实验工作区。
    root = LAB_ROOT / name
    root.mkdir(parents=True, exist_ok=True)
    for relative_path, content in PROJECT_FILES.items():
        (root / relative_path).write_text(content, encoding="utf-8")
    return root

def run_unit_tests(root: Path) -> dict:
    # 运行固定测试入口，不向模型开放任意 Shell。
    result = subprocess.run(
        [sys.executable, "-m", "unittest", "discover", "-s", str(root), "-p", "test_*.py"],
        capture_output=True,
        text=True,
        timeout=10,
    )
    return {
        "passed": result.returncode == 0,
        "returncode": result.returncode,
        "output": (result.stdout + result.stderr)[-2000:],
    }

def search_python(root: Path, query: str) -> list[str]:
    # 返回包含目标文本的文件和行号，忽略 Harness 自己的状态目录。
    matches = []
    for path in sorted(root.rglob("*.py")):
        if ".harness" in path.parts:
            continue
        for line_no, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
            if query in line:
                matches.append(f"{path.relative_to(root)}:{line_no}: {line.strip()}")
    return matches

baseline_root = make_workspace("baseline")
baseline = run_unit_tests(baseline_root)
print({"files": sorted(PROJECT_FILES), "tests_passed": baseline["passed"]})
assert baseline["passed"]

{'files': ['checkout.py', 'pricing.py', 'report.py', 'test_pricing.py'], 'tests_passed': True}


### 6.2 实验一：裸循环为什么会提前结束

假设模型只读取了 `pricing.py`，随后创建 `PricePolicy` 并回复“迁移完成”。应用程序直接相信最终文本。这个流程没有全局搜索，也没有独立验收。

In [3]:
bare_root = make_workspace("bare_loop")

# 模拟模型只修改了定义文件，随后提前宣布完成。
(bare_root / "pricing.py").write_text(
    "class PricePolicy:\n"
    "    def apply(self, amount: float) -> float:\n"
    "        return round(amount * 0.9, 2)\n",
    encoding="utf-8",
)
model_final_text = "迁移完成，PricePolicy 已经替代旧函数。"

test_result = run_unit_tests(bare_root)
remaining_refs = search_python(bare_root, "legacy_price")
print("模型结论:", model_final_text)
print("测试通过:", test_result["passed"])
print("遗留引用数:", len(remaining_refs))
print("首个遗留引用:", remaining_refs[0])

assert model_final_text
assert not test_result["passed"]
assert remaining_refs

模型结论: 迁移完成，PricePolicy 已经替代旧函数。
测试通过: False
遗留引用数: 6
首个遗留引用: checkout.py:1: from pricing import legacy_price


输出证明了两件事：模型的自然语言结论不能充当系统状态；任务完成条件需要由独立程序执行。Harness 因此需要全局状态和 Verifier。

### 6.3 实验二：大结果落盘，摘要进入上下文

工具可能返回数万行日志或完整文档。Harness 直接把全部结果追加到消息历史，会快速消耗上下文。常用处理方式如下：

1. 完整结果写入 Artifact Store。
2. 模型收到摘要、大小、哈希和 `artifact_ref`。
3. 模型需要细节时调用 `read_artifact` 或分段读取工具。

摘要必须披露省略行为，避免模型误以为自己看到了完整内容。

In [4]:
import hashlib

class ArtifactStore:
    def __init__(self, root: Path):
        self.root = root / ".harness" / "artifacts"
        self.root.mkdir(parents=True, exist_ok=True)

    def put_text(self, content: str) -> str:
        digest = hashlib.sha256(content.encode("utf-8")).hexdigest()
        (self.root / f"{digest}.txt").write_text(content, encoding="utf-8")
        return f"artifact://sha256/{digest}"

    def get_text(self, ref: str) -> str:
        digest = ref.rsplit("/", 1)[-1]
        return (self.root / f"{digest}.txt").read_text(encoding="utf-8")

def pack_observation(content: str, store: ArtifactStore, limit: int = 240) -> dict:
    if len(content) <= limit:
        return {"truncated": False, "content": content}
    ref = store.put_text(content)
    return {
        "truncated": True,
        "summary": content[:120] + "...",
        "characters": len(content),
        "artifact_ref": ref,
        "instruction": "需要细节时读取 artifact_ref；当前内容只是摘要。",
    }

artifact_root = make_workspace("artifact_demo")
artifact_store = ArtifactStore(artifact_root)
long_test_log = "test_case passed\n" * 200
packed = pack_observation(long_test_log, artifact_store)
print(json.dumps(packed, ensure_ascii=False, indent=2))

assert packed["truncated"] is True
assert artifact_store.get_text(packed["artifact_ref"]) == long_test_log

{
  "truncated": true,
  "summary": "test_case passed\ntest_case passed\ntest_case passed\ntest_case passed\ntest_case passed\ntest_case passed\ntest_case passed\nt...",
  "characters": 3400,
  "artifact_ref": "artifact://sha256/e636a7ada09e00a101e2b6d5ded216be670e174954b56add0af8b448c2175846",
  "instruction": "需要细节时读取 artifact_ref；当前内容只是摘要。"
}


### 6.4 实验三：显式装配上下文

下面的 Context Builder 把稳定前缀和动态尾部分开。实际系统还会计算 Token、按优先级裁剪、加载仓库规则和检索记忆；这些机制已在第七章展开。

In [5]:
def build_context(task: str, acceptance: list[str], observations: list[dict]) -> list[dict]:
    stable_prefix = (
        "你是代码迁移 Agent。只能通过已声明工具操作工作区。"
        "修改前先搜索影响范围；声明完成前运行测试。"
    )
    dynamic_state = {
        "task": task,
        "acceptance": acceptance,
        "recent_observations": observations[-4:],
    }
    return [
        {"role": "system", "content": stable_prefix},
        {"role": "user", "content": json.dumps(dynamic_state, ensure_ascii=False)},
    ]

context_messages = build_context(
    task="把 legacy_price 迁移到 PricePolicy.apply",
    acceptance=["测试通过", "无 legacy_price 引用"],
    observations=[packed],
)
prefix_hash = hashlib.sha256(context_messages[0]["content"].encode()).hexdigest()[:12]
print("稳定前缀哈希:", prefix_hash)
print("动态状态字段:", list(json.loads(context_messages[1]["content"])))
assert "artifact_ref" in context_messages[1]["content"]

稳定前缀哈希: 3c1502ecba46
动态状态字段: ['task', 'acceptance', 'recent_observations']


### 6.5 实验四：动作先经过 Policy，再进入 Tool Runtime

Policy 的返回值至少包含三类：

- `ALLOW`：直接执行，例如读取工作区文件。
- `DENY`：拒绝执行，例如读取工作区外部路径。
- `REQUIRE_APPROVAL`：暂停并请求审批，例如修改源文件或执行部署。

Policy 判断需要使用结构化动作。依赖自然语言中的“我会小心执行”无法形成安全边界。

In [6]:
from enum import Enum

@dataclass(frozen=True)
class Action:
    action_id: str
    tool: str
    arguments: dict[str, Any]

class Decision(str, Enum):
    ALLOW = "ALLOW"
    DENY = "DENY"
    REQUIRE_APPROVAL = "REQUIRE_APPROVAL"

@dataclass(frozen=True)
class PolicyResult:
    decision: Decision
    reason: str

def resolve_workspace_path(root: Path, relative_path: str) -> Path:
    candidate = (root / relative_path).resolve()
    if not candidate.is_relative_to(root.resolve()):
        raise ValueError("path escapes workspace")
    return candidate

def check_policy(root: Path, action: Action) -> PolicyResult:
    if action.tool not in {"list_files", "read_file", "search_text", "write_file", "run_tests"}:
        return PolicyResult(Decision.DENY, "tool is not in allowlist")
    if "path" in action.arguments:
        try:
            path = resolve_workspace_path(root, action.arguments["path"])
        except ValueError:
            return PolicyResult(Decision.DENY, "path escapes workspace")
        if path.name == ".env" or ".git" in path.parts:
            return PolicyResult(Decision.DENY, "protected path")
    if action.tool == "write_file":
        return PolicyResult(Decision.REQUIRE_APPROVAL, "source write requires approval")
    return PolicyResult(Decision.ALLOW, "read-only or fixed test action")

policy_root = make_workspace("policy_demo")
decisions = {
    "read": check_policy(policy_root, Action("a1", "read_file", {"path": "pricing.py"})),
    "write": check_policy(policy_root, Action("a2", "write_file", {"path": "pricing.py", "content": "..."})),
    "escape": check_policy(policy_root, Action("a3", "read_file", {"path": "../secret.txt"})),
}
print({name: result.decision.value for name, result in decisions.items()})
assert decisions["read"].decision is Decision.ALLOW
assert decisions["write"].decision is Decision.REQUIRE_APPROVAL
assert decisions["escape"].decision is Decision.DENY

{'read': 'ALLOW', 'write': 'REQUIRE_APPROVAL', 'escape': 'DENY'}


### 6.6 Tool Runtime：固定能力面和工作区边界

Notebook 中的 Runtime 只暴露五个领域工具，不开放任意 Shell。路径校验属于应用层防线，无法替代操作系统、容器或虚拟机沙箱。生产环境还要限制进程、网络、文件挂载、凭据和资源用量。[3]

In [7]:
class ToolRuntime:
    def __init__(self, root: Path):
        self.root = root.resolve()
        self.execution_counts: dict[str, int] = {}

    def execute(self, action: Action) -> dict:
        self.execution_counts[action.action_id] = self.execution_counts.get(action.action_id, 0) + 1
        args = action.arguments
        if action.tool == "list_files":
            return {"files": [str(p.relative_to(self.root)) for p in sorted(self.root.glob("*.py"))]}
        if action.tool == "read_file":
            path = resolve_workspace_path(self.root, args["path"])
            return {"path": args["path"], "content": path.read_text(encoding="utf-8")}
        if action.tool == "search_text":
            return {"query": args["query"], "matches": search_python(self.root, args["query"])}
        if action.tool == "write_file":
            path = resolve_workspace_path(self.root, args["path"])
            path.write_text(args["content"], encoding="utf-8")
            return {"path": args["path"], "characters_written": len(args["content"])}
        if action.tool == "run_tests":
            return run_unit_tests(self.root)
        raise ValueError(f"unsupported tool: {action.tool}")

runtime_root = make_workspace("runtime_demo")
runtime = ToolRuntime(runtime_root)
read_result = runtime.execute(Action("read-1", "read_file", {"path": "pricing.py"}))
print(read_result)
assert "legacy_price" in read_result["content"]

{'path': 'pricing.py', 'content': 'def legacy_price(amount: float) -> float:\n    return round(amount * 0.9, 2)\n'}


### 6.7 Event Store：记录事实，重建状态

事件采用追加写入。当前状态可以由事件折叠得到：最后一个终态事件决定运行状态，所有 `tool.completed` 事件构成已经完成的动作集合。

In [8]:
class JsonlEventStore:
    def __init__(self, root: Path):
        self.path = root / ".harness" / "events.jsonl"
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def append(self, event: Event) -> None:
        with self.path.open("a", encoding="utf-8") as file:
            file.write(json.dumps(asdict(event), ensure_ascii=False) + "\n")

    def load(self) -> list[Event]:
        if not self.path.exists():
            return []
        return [Event(**json.loads(line)) for line in self.path.read_text(encoding="utf-8").splitlines()]

event_root = make_workspace("event_demo")
event_store = JsonlEventStore(event_root)
for seq, event_type in enumerate(["run.started", "tool.started", "tool.completed"], 1):
    event_store.append(Event(seq, event_type, "thread-1", "run-1", "turn-1", "action-1"))

loaded_events = event_store.load()
print([event.event_type for event in loaded_events])
assert [event.seq for event in loaded_events] == [1, 2, 3]

['run.started', 'tool.started', 'tool.completed']


### 6.8 实验五：崩溃窗口、幂等与恢复

最危险的时序如下：

```text
1. Harness 发送写入动作
2. 工具完成写入
3. Harness 尚未保存 tool.completed
4. 进程崩溃
5. 恢复程序看到动作未完成，再次执行
```

分布式系统很难对任意外部副作用提供绝对的 exactly-once。工程上常用稳定幂等键、执行回执和状态对账实现“重复请求产生同一结果”。

In [9]:
class ReceiptStore:
    def __init__(self, root: Path):
        self.path = root / ".harness" / "receipts.json"
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def load_all(self) -> dict[str, dict]:
        if not self.path.exists():
            return {}
        return json.loads(self.path.read_text(encoding="utf-8"))

    def get(self, action_id: str) -> dict | None:
        return self.load_all().get(action_id)

    def put(self, action_id: str, result: dict) -> None:
        receipts = self.load_all()
        receipts[action_id] = result
        self.path.write_text(json.dumps(receipts, ensure_ascii=False, indent=2), encoding="utf-8")

def execute_once(action: Action, runtime: ToolRuntime, receipts: ReceiptStore) -> tuple[dict, bool]:
    # 命中执行回执时直接复用结果，避免恢复后重复副作用。
    cached = receipts.get(action.action_id)
    if cached is not None:
        return cached, True
    result = runtime.execute(action)
    receipts.put(action.action_id, result)
    return result, False

In [10]:
crash_root = make_workspace("crash_recovery")
crash_runtime = ToolRuntime(crash_root)
crash_receipts = ReceiptStore(crash_root)
crash_events = JsonlEventStore(crash_root)

migration_action = Action(
    "migrate-pricing-v1",
    "write_file",
    {
        "path": "pricing.py",
        "content": "class PricePolicy:\n    def apply(self, amount: float) -> float:\n        return round(amount * 0.9, 2)\n",
    },
)

crash_events.append(Event(1, "tool.started", "thread-crash", "run-1", "turn-1", migration_action.action_id))
first_result, first_cached = execute_once(migration_action, crash_runtime, crash_receipts)
# 此处模拟进程在写入完成后、tool.completed 事件落盘前崩溃。

resumed_result, resumed_cached = execute_once(migration_action, crash_runtime, crash_receipts)
crash_events.append(Event(2, "tool.completed", "thread-crash", "run-2", "turn-1", migration_action.action_id, resumed_result))

print({
    "first_cached": first_cached,
    "resumed_from_receipt": resumed_cached,
    "physical_executions": crash_runtime.execution_counts[migration_action.action_id],
})
assert first_result == resumed_result
assert resumed_cached is True
assert crash_runtime.execution_counts[migration_action.action_id] == 1

{'first_cached': False, 'resumed_from_receipt': True, 'physical_executions': 1}


### 6.9 实验六：Supervisor 控制预算、超时和取消

Supervisor 在每轮开始前检查预算，在工具执行前检查取消信号，在可重试错误后计算退避时间。下面只观察预算和取消两个确定性分支。

In [11]:
import time

@dataclass
class RunBudget:
    max_model_calls: int
    max_tool_calls: int
    deadline_monotonic: float
    model_calls: int = 0
    tool_calls: int = 0
    cancelled: bool = False

    def before_model(self) -> None:
        if self.cancelled:
            raise RuntimeError("run cancelled")
        if time.monotonic() >= self.deadline_monotonic:
            raise TimeoutError("run deadline exceeded")
        if self.model_calls >= self.max_model_calls:
            raise RuntimeError("model call budget exhausted")
        self.model_calls += 1

    def before_tool(self) -> None:
        if self.cancelled:
            raise RuntimeError("run cancelled")
        if self.tool_calls >= self.max_tool_calls:
            raise RuntimeError("tool call budget exhausted")
        self.tool_calls += 1

budget = RunBudget(max_model_calls=2, max_tool_calls=1, deadline_monotonic=time.monotonic() + 30)
budget.before_model()
budget.before_tool()
budget.cancelled = True
try:
    budget.before_model()
except RuntimeError as error:
    cancellation_result = str(error)

print({"model_calls": budget.model_calls, "tool_calls": budget.tool_calls, "result": cancellation_result})
assert cancellation_result == "run cancelled"

{'model_calls': 1, 'tool_calls': 1, 'result': 'run cancelled'}


### 6.10 实验七：Verifier 决定系统是否完成

Verifier 使用任务开始前确定的验收标准。它不读取模型的自我评价，只读取工作区和测试结果。

In [12]:
def verify_migration(root: Path) -> dict:
    tests = run_unit_tests(root)
    legacy_refs = search_python(root, "legacy_price")
    checks = {
        "tests_passed": tests["passed"],
        "legacy_refs_removed": len(legacy_refs) == 0,
    }
    return {
        "passed": all(checks.values()),
        "checks": checks,
        "legacy_refs": legacy_refs,
        "test_output": tests["output"],
    }

bare_verification = verify_migration(bare_root)
baseline_verification = verify_migration(baseline_root)
print("裸循环:", bare_verification["checks"])
print("基线项目:", baseline_verification["checks"])
assert not bare_verification["passed"]
assert baseline_verification["checks"]["tests_passed"]
assert not baseline_verification["checks"]["legacy_refs_removed"]

裸循环: {'tests_passed': False, 'legacy_refs_removed': False}
基线项目: {'tests_passed': True, 'legacy_refs_removed': False}


### 6.11 实验八：组装一个可恢复的最小 Harness

确定性动作序列用于稳定观察运行时，不代表模型生成结果。每个动作仍然经过 Policy、Approval、幂等执行、事件记录和最终验收。

In [13]:
MIGRATED_FILES = {
    "pricing.py": '''class PricePolicy:\n    def apply(self, amount: float) -> float:\n        return round(amount * 0.9, 2)\n''',
    "checkout.py": '''from pricing import PricePolicy\n\ndef checkout_total(amount: float) -> float:\n    return PricePolicy().apply(amount)\n''',
    "report.py": '''from pricing import PricePolicy\n\ndef display_price(amount: float) -> str:\n    return f"¥{PricePolicy().apply(amount):.2f}"\n''',
    "test_pricing.py": '''import unittest\nfrom pricing import PricePolicy\nfrom checkout import checkout_total\nfrom report import display_price\n\nclass PricingTest(unittest.TestCase):\n    def test_discount(self):\n        self.assertEqual(PricePolicy().apply(100), 90.0)\n\n    def test_checkout(self):\n        self.assertEqual(checkout_total(200), 180.0)\n\n    def test_report(self):\n        self.assertEqual(display_price(50), "¥45.00")\n\nif __name__ == "__main__":\n    unittest.main()\n''',
}

scripted_actions = [
    Action("search-impact", "search_text", {"query": "legacy_price"}),
    *[
        Action(f"write-{path}", "write_file", {"path": path, "content": content})
        for path, content in MIGRATED_FILES.items()
    ],
    Action("run-tests", "run_tests", {}),
    Action("search-remains", "search_text", {"query": "legacy_price"}),
]
print([action.tool for action in scripted_actions])

['search_text', 'write_file', 'write_file', 'write_file', 'write_file', 'run_tests', 'search_text']


In [14]:
def run_offline_harness(root: Path, actions: list[Action]) -> dict:
    runtime = ToolRuntime(root)
    receipts = ReceiptStore(root)
    events = JsonlEventStore(root)
    thread_id, run_id, turn_id = "thread-migration", "run-offline", "turn-1"
    seq = 0

    def record(event_type: str, action: Action | None = None, payload: dict | None = None) -> None:
        nonlocal seq
        seq += 1
        events.append(Event(seq, event_type, thread_id, run_id, turn_id, action.action_id if action else None, payload or {}))

    record("run.started")
    for action in actions:
        policy = check_policy(root, action)
        record("policy.decided", action, {"decision": policy.decision.value, "reason": policy.reason})
        if policy.decision is Decision.DENY:
            record("run.failed", action, {"reason": policy.reason})
            break
        if policy.decision is Decision.REQUIRE_APPROVAL:
            # 教学环境只批准工作区内的 Python 源文件写入。
            record("approval.granted", action, {"actor": "lab-owner"})
        record("tool.started", action)
        result, from_receipt = execute_once(action, runtime, receipts)
        record("tool.completed", action, {"result": result, "from_receipt": from_receipt})

    verification = verify_migration(root)
    record("verification.completed", payload=verification["checks"])
    record("run.succeeded" if verification["passed"] else "run.failed")
    return {"verification": verification, "events": events.load(), "runtime": runtime}

offline_root = make_workspace("offline_harness")
offline_run = run_offline_harness(offline_root, scripted_actions)
event_counts = {}
for event in offline_run["events"]:
    event_counts[event.event_type] = event_counts.get(event.event_type, 0) + 1

print({"verification": offline_run["verification"]["checks"], "event_counts": event_counts})
assert offline_run["verification"]["passed"]
assert event_counts["approval.granted"] == 4
assert event_counts["run.succeeded"] == 1

{'verification': {'tests_passed': True, 'legacy_refs_removed': True}, 'event_counts': {'run.started': 1, 'policy.decided': 7, 'tool.started': 7, 'tool.completed': 7, 'approval.granted': 4, 'verification.completed': 1, 'run.succeeded': 1}}


## 7 完整实验：使用 DeepSeek 驱动真实 Harness

### 7.1 环境变量

真实实验使用 OpenAI-compatible Chat Completions 和 Function Calling。仓库根目录 `.env` 可以配置：

```dotenv
LLM_API_KEY=你的密钥
LLM_BASE_URL=https://api.deepseek.com
LLM_MODEL=deepseek-v4-pro
RUN_LIVE_LLM=1
```

Notebook 使用 `find_dotenv(usecwd=True)` 向上定位 `.env`，并采用 `override=False`：Shell 或 Notebook 进程中显式设置的变量优先。代码不打印 API Key。仓库验证时强制 `RUN_LIVE_LLM=0`。

In [15]:
import os
from dotenv import find_dotenv, load_dotenv

ENV_PATH = find_dotenv(usecwd=True)
if ENV_PATH:
    load_dotenv(ENV_PATH, override=False)

RUN_LIVE_LLM = os.getenv("RUN_LIVE_LLM", "0") == "1"
LLM_API_KEY = (os.getenv("LLM_API_KEY") or os.getenv("DEEPSEEK_API_KEY") or os.getenv("OPENAI_API_KEY") or "").strip()
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "https://api.deepseek.com").strip()
LLM_MODEL = os.getenv("LLM_MODEL", "deepseek-chat").strip()

print({
    "env_loaded": bool(ENV_PATH),
    "run_live_llm": RUN_LIVE_LLM,
    "api_key_configured": bool(LLM_API_KEY),
    "base_url": LLM_BASE_URL,
    "model": LLM_MODEL,
})

{'env_loaded': True, 'run_live_llm': False, 'api_key_configured': True, 'base_url': 'https://api.deepseek.com', 'model': 'deepseek-v4-pro'}


### 7.2 工具协议

Schema 告诉模型可以请求哪些动作。Harness 仍然掌握最终执行权。`write_file` 要求完整文件内容，避免补丁协议本身干扰本章主题。

In [16]:
TOOL_SCHEMAS = [
    {"type": "function", "function": {"name": "list_files", "description": "列出工作区中的 Python 文件", "parameters": {"type": "object", "properties": {}, "additionalProperties": False}}},
    {"type": "function", "function": {"name": "read_file", "description": "读取工作区内一个 UTF-8 文本文件", "parameters": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"], "additionalProperties": False}}},
    {"type": "function", "function": {"name": "search_text", "description": "在全部 Python 文件中搜索文本并返回文件、行号和内容", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"], "additionalProperties": False}}},
    {"type": "function", "function": {"name": "write_file", "description": "完整覆盖工作区内一个 UTF-8 文本文件", "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"], "additionalProperties": False}}},
    {"type": "function", "function": {"name": "run_tests", "description": "运行项目固定的 unittest 测试套件", "parameters": {"type": "object", "properties": {}, "additionalProperties": False}}},
]
print([item["function"]["name"] for item in TOOL_SCHEMAS])
assert len(TOOL_SCHEMAS) == 5

['list_files', 'read_file', 'search_text', 'write_file', 'run_tests']


### 7.3 一轮多个 Tool Call：先定义批次语义

**同一轮返回多个 Tool Call，只代表模型一次提出了多个调用请求。** OpenAI-compatible 协议使用 `tool_call.id` 关联请求与结果，不提供 `depends_on`、事务组、回滚和原子提交。

模型生成这一批调用时，还没有看到其中任何一个调用的执行结果。因此，第二个调用不能使用第一个调用运行后才产生的数据。存在运行时依赖的动作必须拆到下一轮，或由 Harness / Workflow 提供显式 DAG。

本章采用两种批次模式：

| 批次内容 | 执行模式 | 前一个调用失败后的处理 |
| --- | --- | --- |
| 全部是 `list_files`、`read_file`、`search_text` | `BEST_EFFORT_READS` | 继续其他独立读取 |
| 包含 `write_file` 或 `run_tests` | `ORDERED_FAIL_FAST` | 跳过剩余调用 |

`ORDERED_FAIL_FAST` 是保守默认值。批次中出现副作用或后置检查时，Harness 按顺序执行；任一步失败、被 Policy 拒绝或未获批准，后续调用全部返回 `SKIPPED_PRIOR_ACTION_FAILED`。

```mermaid
flowchart TD
    M["模型返回一批 Tool Call"] --> C{"是否全部只读"}
    C -- "是" --> R["BEST_EFFORT_READS"]
    C -- "否" --> O["ORDERED_FAIL_FAST"]
    R --> RE["逐个执行；单个失败不阻断其他读取"]
    O --> E["顺序执行当前调用"]
    E --> S{"执行成功"}
    S -- "是" --> N{"还有调用"}
    N -- "是" --> E
    N -- "否" --> B["回传全部 Tool Result"]
    S -- "否" --> K["剩余调用标记 SKIPPED"]
    K --> B
    RE --> B
    B --> L["进入下一轮模型推理"]
```

跳过的 Tool Call 仍需返回 Tool Result。API 消息历史中的每个 `tool_call.id` 都要有对应结果，区别只在于 Harness 没有执行该工具。


In [17]:
from enum import Enum


class ToolEffect(str, Enum):
    READ_ONLY = "READ_ONLY"
    SIDE_EFFECT = "SIDE_EFFECT"
    CHECK = "CHECK"


TOOL_EFFECTS = {
    "list_files": ToolEffect.READ_ONLY,
    "read_file": ToolEffect.READ_ONLY,
    "search_text": ToolEffect.READ_ONLY,
    "write_file": ToolEffect.SIDE_EFFECT,
    "run_tests": ToolEffect.CHECK,
}


def choose_batch_mode(tool_names: list[str]) -> str:
    '''只读批次允许 best-effort；包含写入或检查时按顺序失败即停。'''
    effects = [TOOL_EFFECTS.get(name, ToolEffect.SIDE_EFFECT) for name in tool_names]
    if all(effect is ToolEffect.READ_ONLY for effect in effects):
        return "BEST_EFFORT_READS"
    return "ORDERED_FAIL_FAST"


def skipped_tool_result(blocked_by: str) -> dict:
    '''未执行也要产生 Tool Result，保持消息协议完整。'''
    return {
        "ok": False,
        "status": "skipped",
        "error_code": "SKIPPED_PRIOR_ACTION_FAILED",
        "blocked_by": blocked_by,
    }


def next_blocked_by(
    batch_mode: str,
    current_blocker: str | None,
    action_id: str,
    result: dict,
) -> str | None:
    '''有序批次首次失败后，固定返回导致失败的 action_id。'''
    if current_blocker is not None:
        return current_blocker
    if batch_mode == "ORDERED_FAIL_FAST" and result.get("ok") is False:
        return action_id
    return None


print({
    "三个读取": choose_batch_mode(["list_files", "read_file", "search_text"]),
    "读取后写入": choose_batch_mode(["read_file", "write_file"]),
    "写入后测试": choose_batch_mode(["write_file", "run_tests"]),
})

assert choose_batch_mode(["read_file", "search_text"]) == "BEST_EFFORT_READS"
assert choose_batch_mode(["read_file", "write_file"]) == "ORDERED_FAIL_FAST"
assert skipped_tool_result("action-1")["blocked_by"] == "action-1"

failed_result = {"ok": False, "error_code": "TOOL_EXECUTION_FAILED"}
ordered_blocker = next_blocked_by(
    "ORDERED_FAIL_FAST", None, "write-pricing", failed_result
)
second_write_result = skipped_tool_result(ordered_blocker)
read_only_blocker = next_blocked_by(
    "BEST_EFFORT_READS", None, "read-missing", failed_result
)

print({
    "第一个写入失败": ordered_blocker,
    "第二个写入结果": second_write_result,
    "只读失败后的阻断值": read_only_blocker,
})
assert second_write_result["status"] == "skipped"
assert second_write_result["blocked_by"] == "write-pricing"
assert read_only_blocker is None


{'三个读取': 'BEST_EFFORT_READS', '读取后写入': 'ORDERED_FAIL_FAST', '写入后测试': 'ORDERED_FAIL_FAST'}
{'第一个写入失败': 'write-pricing', '第二个写入结果': {'ok': False, 'status': 'skipped', 'error_code': 'SKIPPED_PRIOR_ACTION_FAILED', 'blocked_by': 'write-pricing'}, '只读失败后的阻断值': None}


### 7.4 真实运行循环

下面的循环按照六个阶段运行：

1. 构造消息并调用模型。
2. 将本轮 Tool Call 分类为只读批次或有序批次。
3. 为每个调用生成稳定 `action_id`，执行 Policy 和审批检查。
4. 执行工具并先保存 Receipt 与 Event。
5. 前序动作失败时，跳过有序批次的剩余调用，但为每个调用返回结构化 Tool Result。
6. 模型停止调用工具后，由 Verifier 独立验收。

实验中的“审批”是本地规则回调，只允许沙箱工作区内的 `.py` 文件。真实人工审批至少包含 `PENDING`、`APPROVED`、`DENIED` 三种状态：等待时保存 Checkpoint，批准后恢复，人工拒绝后结束当前 Run 或转人工接管。

```mermaid
flowchart LR
    C["Context"] --> M["Model"]
    M --> BT["Batch Type"]
    BT --> P["Policy / Approval"]
    P --> T["Tool Runtime"]
    T --> PR["Receipt + Event"]
    PR --> M
    M --> V["Verifier"]
    V -- "FAIL" --> M
    V -- "PASS" --> D["SUCCEEDED"]
```


In [18]:
def run_live_harness(root: Path, max_rounds: int = 12) -> dict:
    from openai import OpenAI

    client_options = {"api_key": LLM_API_KEY}
    if LLM_BASE_URL:
        client_options["base_url"] = LLM_BASE_URL
    client = OpenAI(**client_options)

    runtime = ToolRuntime(root)
    event_store = JsonlEventStore(root)
    artifact_store = ArtifactStore(root)
    receipt_store = ReceiptStore(root)
    thread_id = f"thread-{uuid.uuid4().hex[:8]}"
    run_id = f"run-{uuid.uuid4().hex[:8]}"
    turn_id = "turn-1"
    seq = 0
    tool_calls_total = 0

    def record(
        event_type: str,
        action_id: str | None = None,
        payload: dict | None = None,
    ) -> None:
        nonlocal seq
        seq += 1
        event_store.append(
            Event(
                seq,
                event_type,
                thread_id,
                run_id,
                turn_id,
                action_id,
                payload or {},
            )
        )

    def append_tool_result(messages: list[dict], tool_call_id: str, result: dict) -> None:
        # 每个 Tool Call 都必须有对应 Tool Result，包括被跳过的调用。
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call_id,
            "content": json.dumps(result, ensure_ascii=False),
        })

    messages = [
        {
            "role": "system",
            "content": (
                "你是代码迁移 Agent。所有文件操作必须使用工具。先列出文件并搜索影响范围，再读取相关文件。"
                "把 legacy_price(amount) 完整迁移到 PricePolicy.apply(amount)，同步修改生产代码和测试。"
                "存在数据依赖的写操作分轮请求；不要把依赖前序结果的动作放在同一批 Tool Call 中。"
                "声明完成前必须运行测试并再次搜索 legacy_price。遇到失败时根据工具结果继续修复。"
            ),
        },
        {
            "role": "user",
            "content": "完成迁移。验收条件：全部 unittest 通过；所有 Python 文件中没有 legacy_price。",
        },
    ]
    record("run.started")

    for round_no in range(1, max_rounds + 1):
        record("model.started", payload={"round": round_no})
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
        )
        message = response.choices[0].message
        tool_calls = message.tool_calls or []
        record(
            "model.completed",
            payload={"round": round_no, "tool_calls": len(tool_calls)},
        )

        if tool_calls:
            messages.append(message.model_dump(exclude_none=True))
            tool_names = [call.function.name for call in tool_calls]
            batch_mode = choose_batch_mode(tool_names)
            blocked_by: str | None = None
            record(
                "tool_batch.started",
                payload={"mode": batch_mode, "tools": tool_names},
            )

            for call_index, tool_call in enumerate(tool_calls, 1):
                tool_calls_total += 1
                arguments = json.loads(tool_call.function.arguments or "{}")

                # tool_call.id 负责 API 消息关联；action_id 负责幂等和审计。
                canonical_action = json.dumps(
                    arguments,
                    ensure_ascii=False,
                    sort_keys=True,
                )
                action_digest = hashlib.sha256(
                    (
                        f"{thread_id}:{round_no}:{call_index}:"
                        f"{tool_call.function.name}:{canonical_action}"
                    ).encode()
                ).hexdigest()[:16]
                action = Action(
                    f"action-{action_digest}",
                    tool_call.function.name,
                    arguments,
                )

                # 有序批次已经失败时，剩余调用只回传 skipped，不进入 Runtime。
                if blocked_by is not None:
                    result = skipped_tool_result(blocked_by)
                    record(
                        "tool.skipped",
                        action.action_id,
                        {
                            "reason": result["error_code"],
                            "blocked_by": blocked_by,
                        },
                    )
                    append_tool_result(messages, tool_call.id, result)
                    continue

                policy = check_policy(root, action)
                record(
                    "policy.decided",
                    action.action_id,
                    {
                        "decision": policy.decision.value,
                        "reason": policy.reason,
                    },
                )

                if policy.decision is Decision.DENY:
                    result = {
                        "ok": False,
                        "status": "denied",
                        "error_code": "POLICY_DENIED",
                        "error": policy.reason,
                    }
                else:
                    if policy.decision is Decision.REQUIRE_APPROVAL:
                        # 教学回调：真实系统需要持久化 PENDING / APPROVED / DENIED。
                        approved = arguments.get("path", "").endswith(".py")
                        record(
                            "approval.granted" if approved else "approval.denied",
                            action.action_id,
                            {"source": "lab-rule"},
                        )
                        if not approved:
                            result = {
                                "ok": False,
                                "status": "denied",
                                "error_code": "APPROVAL_DENIED",
                                "error": "lab approval rule rejected this action",
                            }
                        else:
                            result = None
                    else:
                        result = None

                    if result is None:
                        try:
                            record(
                                "tool.started",
                                action.action_id,
                                {"tool": action.tool},
                            )
                            raw_result, from_receipt = execute_once(
                                action,
                                runtime,
                                receipt_store,
                            )
                            observation = pack_observation(
                                json.dumps(raw_result, ensure_ascii=False),
                                artifact_store,
                                limit=3000,
                            )
                            result = {
                                "ok": True,
                                "status": "succeeded",
                                **observation,
                            }
                            record(
                                "tool.completed",
                                action.action_id,
                                {
                                    "tool": action.tool,
                                    "truncated": observation["truncated"],
                                    "from_receipt": from_receipt,
                                },
                            )
                        except Exception as exc:
                            result = {
                                "ok": False,
                                "status": "failed",
                                "error_code": "TOOL_EXECUTION_FAILED",
                                "error": str(exc),
                            }
                            record(
                                "tool.failed",
                                action.action_id,
                                {
                                    "tool": action.tool,
                                    "error": str(exc),
                                },
                            )

                blocked_by = next_blocked_by(
                    batch_mode,
                    blocked_by,
                    action.action_id,
                    result,
                )

                append_tool_result(messages, tool_call.id, result)

            record(
                "tool_batch.completed",
                payload={"mode": batch_mode, "blocked_by": blocked_by},
            )
            # 当前批次的结果已经加入 messages，进入下一轮模型推理。
            continue

        messages.append(message.model_dump(exclude_none=True))
        verification = verify_migration(root)
        record("verification.completed", payload=verification["checks"])
        if verification["passed"]:
            record("run.succeeded")
            return {
                "passed": True,
                "rounds": round_no,
                "tool_calls": tool_calls_total,
                "final_text": message.content or "",
                "verification": verification,
                "events": event_store.load(),
            }

        # 把确定性验收失败作为新观察返回，模型可以继续修复。
        messages.append({
            "role": "user",
            "content": (
                "系统验收未通过，请继续修复："
                + json.dumps(verification["checks"], ensure_ascii=False)
            ),
        })

    record("run.failed", payload={"reason": "round budget exhausted"})
    return {
        "passed": False,
        "rounds": max_rounds,
        "tool_calls": tool_calls_total,
        "verification": verify_migration(root),
        "events": event_store.load(),
    }


In [19]:
if RUN_LIVE_LLM and LLM_API_KEY:
    live_root = make_workspace("live_deepseek")
    live_run = run_live_harness(live_root)
    print({
        "passed": live_run["passed"],
        "rounds": live_run["rounds"],
        "tool_calls": live_run["tool_calls"],
        "verification": live_run["verification"]["checks"],
    })
    assert live_run["passed"], live_run["verification"]
else:
    live_run = None
    print("live harness skipped: set RUN_LIVE_LLM=1 and configure API credentials")

live harness skipped: set RUN_LIVE_LLM=1 and configure API credentials


### 7.5 如何解读真实实验

真实模型负责选择读取、搜索、写入和测试动作；Harness 始终负责以下工作：

- 将动作限制在已声明工具和实验工作区内。
- 将本轮 Tool Call 分类为只读批次或有序批次。
- 前序动作失败时跳过后续副作用，并返回 `SKIPPED_PRIOR_ACTION_FAILED`。
- 对写入动作执行审批回调。
- 为每次模型调用、批次、策略判断和工具执行写入事件。
- 对大结果保存完整 Artifact 并回传摘要。
- 在模型停止调用工具后独立运行验收。
- 验收失败时把结构化失败信息送回下一轮。

一个批次中出现 `tool.skipped` 表示 Harness 主动保护了依赖链，工具函数没有被执行。模型在下一轮看到失败和跳过原因后重新规划。

本章的多文件修改发生在隔离工作区。中间文件允许短暂不一致，Verifier 通过后的 Git Commit 或发布动作才是对外 Commit Point。真实生产还需增加 OS 级沙箱、持久化任务队列、租户隔离、密钥代理和更严格的重试策略。


## 8 生产机制

### 8.1 权限与沙箱分层

| 层级 | 典型控制 | 解决的问题 |
| --- | --- | --- |
| Tool Schema | 只暴露允许的动作和参数 | 缩小模型动作空间 |
| Policy Engine | 用户、租户、资源、风险和时间规则 | 业务授权 |
| Approval | 高风险动作由人确认 | 关键副作用兜底 |
| Process Sandbox | 用户、进程、文件、CPU、内存限制 | 限制本机影响范围 |
| Network Policy | 域名、端口、协议、流量限制 | 防止任意外联和数据泄漏 |
| Credential Broker | 短期凭据、按工具注入、禁止进入 Prompt | 控制密钥暴露 |
| Audit Log | 记录主体、动作、参数摘要和结果 | 追责与合规 |

模型不应直接看到长期 API Key。Harness 在工具执行阶段按身份和资源注入短期凭据，并对日志做脱敏。

### 8.2 重试需要先分类

| 失败类型 | 例子 | 处理方式 |
| --- | --- | --- |
| 瞬时基础设施失败 | 429、连接重置、Worker 暂时不可用 | 指数退避、抖动、有限重试 |
| 确定性输入失败 | Schema 校验失败、文件不存在 | 把错误作为观察交给模型修正 |
| 权限失败 | Policy Deny、审批拒绝 | 立即停止当前动作，不自动绕过 |
| 副作用结果未知 | 请求超时，但外部系统可能已执行 | 使用幂等键查询结果，再决定重试 |
| 验收失败 | 测试失败、遗留引用 | 返回 Verifier 证据，允许继续修复 |
| 预算耗尽 | 轮数、Token、费用达到上限 | 保存 Checkpoint，停止或请求人工处理 |

对所有错误统一重试会放大副作用和费用。重试策略需要绑定错误分类、动作风险和幂等能力。

### 8.3 并发、恢复和一致性

多 Worker 场景至少需要以下机制：

1. 同一个 Run 同时只有一个所有者，可用租约或乐观锁实现。
2. Checkpoint 保存可恢复状态：最近事件序号、待处理动作、预算使用量和 Artifact 引用。
3. Worker 恢复时先读取事件和执行回执，再决定继续、对账或补偿。
4. Tool Result 与事件写入尽量放在同一事务；外部系统无法加入事务时使用 Outbox、幂等键或补偿动作。
5. `cancel_requested` 与 `cancelled` 分开：前者是请求，后者表示 Worker 已到达安全停止点。

Fork 任务时，新 Run 继承指定 Checkpoint 之前的事件和工作区快照，随后使用新的 `run_id`。两个分支的后续动作、预算和审批记录独立保存。

### 8.4 什么算跨系统

**一次业务操作需要修改两个或更多独立提交的资源，调用方又无法用一个本地 ACID 事务统一提交和回滚，这次操作就跨越了事务边界。**

判断依据是事务控制权，不是机器数量、域名数量或团队数量：

1. 能否用同一个数据库连接开启一次事务。
2. 一次 `COMMIT` 能否同时提交全部修改。
3. 任一步失败时，一次 `ROLLBACK` 能否撤销已经完成的修改。
4. 调用方是否能控制每个参与者的提交时机。

| 场景 | 是否跨系统 | 原因 | 主要方案 |
| --- | --- | --- | --- |
| 同一 PostgreSQL 中更新订单表和库存表 | 否 | 同一个本地事务可以提交或回滚 | 数据库事务 |
| 订单数据库 + 第三方支付 API | 是 | 支付平台已经独立提交，订单库无法回滚支付 | Saga + 幂等退款 |
| 业务数据库 + Kafka 消息 | 是 | 数据库提交和消息发送是两个提交点 | Transactional Outbox |
| Jira 更新任务 + 飞书发送通知 | 是 | 两个平台各自保存状态 | Saga、对账或部分成功 |
| 同一台机器上的数据库 + 文件系统 | 是 | 数据库事务通常不能回滚文件写入 | 临时文件、原子替换、补偿 |
| 通过库存服务 API 修改与订单服务共用的数据库 | 是 | 调用方只能控制 API，不能把远端提交加入当前事务 | 服务级 Saga 或重新划分边界 |

“两个工具”不一定跨系统。两个工具都通过同一事务对象更新同一数据库时，可以保持本地原子性。一个工具同时调用支付、库存和消息平台时，虽然模型只看到一个 Tool Call，工具内部仍然跨越多个事务边界。


### 8.5 Saga：用本地事务和补偿维持跨系统一致性

Saga 把一次跨系统业务操作拆成一组按顺序执行的本地事务。每个参与系统独立提交；后续步骤失败时，协调器调用已经完成步骤的补偿动作，使业务最终回到可接受状态。[10][11]

订单示例：

| 顺序 | 正向动作 | 本地提交 | 补偿动作 |
| --- | --- | --- | --- |
| 1 | 创建订单 `PENDING` | 订单库 | 取消订单 |
| 2 | 预留库存 | 库存库 | 释放库存 |
| 3 | 扣款 | 支付平台 | 发起退款 |
| 4 | 创建发货单 | 物流平台 | 取消发货单 |

假设第 4 步失败，协调器依次发起退款、释放库存、取消订单。退款是新的业务动作，原扣款记录仍然存在；因此补偿和数据库 `ROLLBACK` 的语义不同。

```mermaid
sequenceDiagram
    participant H as Harness / Saga Coordinator
    participant O as Order Service
    participant I as Inventory Service
    participant P as Payment Provider
    participant S as Shipping Service

    H->>O: create_order(idempotency_key)
    O-->>H: order=PENDING
    H->>I: reserve_stock(idempotency_key)
    I-->>H: reserved
    H->>P: charge(idempotency_key)
    P-->>H: charged
    H->>S: create_shipment(idempotency_key)
    S-->>H: FAILED
    H->>P: refund(original_payment_id)
    P-->>H: refund accepted
    H->>I: release_stock(reservation_id)
    I-->>H: released
    H->>O: cancel_order(order_id)
    O-->>H: order=CANCELLED
```

Saga 需要保存以下状态：

- `saga_id`、当前步骤和业务状态。
- 每一步的幂等键、请求摘要和执行回执。
- 已完成步骤及对应补偿动作。
- 重试次数、最后错误和下一次执行时间。
- 补偿失败后的告警与人工接管状态。

Agent 场景同样存在 Saga。假设 Agent 先把 Jira Issue 改为 `Done`，随后发送飞书通知失败：协调器可以把 Jira 恢复为原状态，也可以保留 `Done` 并进入“通知待补发”。飞书消息发送成功后通常无法真正撤回，补偿动作可能是发送更正消息。业务必须提前定义每一步是否可补偿、补偿内容和不可逆的 Pivot 点。[12]

Saga 提供最终一致性，执行期间允许出现中间状态。补偿动作也可能失败，所以每一步都要幂等、可重试、可对账。单数据库事务足够时优先使用本地事务；跨系统步骤没有业务补偿语义时，需要重新设计流程或加入人工门禁。


### 8.6 Agent 可读环境

Harness 需要把环境反馈转换为模型可以定位和处理的信息：[4][9]

- 测试失败包含文件、行号、期望值和实际值。
- 日志包含时间、级别、组件、请求 ID 和错误类型。
- UI 状态提供文本或结构化快照，避免只给模糊截图。
- 仓库规则放在稳定、分层、可发现的文档中。
- 工具输出保持短小，完整结果通过 Artifact 引用读取。
- 验收命令可重复运行，结果不依赖人工解释。

环境反馈越清晰，模型花在猜测系统状态上的调用越少。


### 8.7 最小可观测字段

每个 Run 至少记录：

| 类别 | 字段 |
| --- | --- |
| 身份 | tenant、user、thread、run、turn |
| 模型 | provider、model、request_id、latency、tokens、cache |
| 动作 | action_id、tool、参数摘要、policy、approval、duration |
| 状态 | started、paused、resumed、cancelled、failed、succeeded |
| 结果 | verifier checks、artifact refs、error class |
| 成本 | 模型调用数、工具调用数、Token、费用、人工介入次数 |

本章事件日志只演示协议。第十二章会继续实现 Trace、评测集、Guardrails 指标和人审队列。


## 9 When：什么时候需要 Harness

### 9.1 场景判断

| 场景 | 推荐实现 |
| --- | --- |
| 单轮问答、分类、结构化提取 | 直接调用模型，保留基础超时和日志 |
| 少量只读工具、任务在数秒内完成 | 轻量 Agent Loop + 工具白名单 |
| 多步骤文件或代码操作 | 基础 Harness + 工作区 + 事件 + Verifier |
| 写业务数据、发消息、部署、付款 | Policy + Approval + Idempotency + Audit |
| 长任务、客户端可能断开 | 持久化 Thread + Worker + Checkpoint + Resume |
| 多用户生产服务 | 租户隔离 + 身份授权 + 密钥代理 + 并发控制 |
| 高合规或高损失场景 | 强沙箱、人工门禁、完整追踪和严格评测 |

判断强度主要看四个变量：任务持续时间、外部副作用、失败成本和并发用户数。

### 9.2 自研、框架和现成运行时

| 选择 | 适用条件 | 主要代价 |
| --- | --- | --- |
| 轻量自研 | 流程短、工具少、团队需要完全控制协议 | 状态、恢复和观测代码由团队维护 |
| Agent Framework | 需要图状态、持久化、工具集成或多种模型适配 | 需要理解框架状态语义和升级成本 |
| 托管 Agent Runtime | 希望快速获得沙箱、队列、会话和扩缩容 | 平台绑定、成本、数据边界和定制限制 |
| 混合方案 | 核心状态自持有，执行或模型调用使用托管服务 | 系统边界和故障责任更复杂 |

Harness 是一组职责。团队可以组合现成组件，不需要从零实现每个模块。第十一章会用同一技术场景比较具体框架。

## 10 Trade-off、踩坑与检查表

### 10.1 常见取舍

| 决策 | 方案 A | 方案 B | 选择依据 |
| --- | --- | --- | --- |
| 状态 | 当前状态表 | 追加事件日志 | 查询效率与审计恢复需求 |
| 执行位置 | 本地进程 | 容器或 VM | 风险、隔离、启动延迟和成本 |
| 工具集 | 固定白名单 | 动态发现 | 可控性与扩展速度 |
| 权限 | 自动执行 | 人工审批 | 动作可逆性和损失上限 |
| 输出 | 全量进入上下文 | Artifact + 摘要 | 信息完整度与 Token 成本 |
| 自主性 | 模型自由规划 | 确定性状态机 | 任务开放度与合规要求 |
| 恢复 | 从头重跑 | Checkpoint 续跑 | 副作用、任务时长和实现复杂度 |

状态机适合固定审批流，模型适合开放分析和计划。生产 Agent 通常把确定性控制交给 Harness，把开放判断交给模型。

### 10.2 十二个常见坑

1. 把模型最终文本直接写成 `SUCCEEDED`。
2. 在内存中保存全部状态，进程重启后无法恢复。
3. 工具执行成功后未保存回执，重试造成重复副作用。
4. 只校验工具名称，没有校验路径、资源和调用主体。
5. 把 API Key、Cookie 或数据库密码放进 Prompt。
6. 把完整日志持续追加进上下文，导致成本上升和注意力稀释。
7. 对权限拒绝、参数错误和网络超时使用同一重试策略。
8. Checkpoint 只保存消息，没有保存工作区、预算和待处理动作。
9. 多 Worker 同时推进同一个 Run，产生交叉写入。
10. Trace 记录了大量文本，却缺少稳定 ID、错误分类和验收结果。
11. 前序 Tool Call 失败后继续执行后续副作用，没有返回 `skipped` 结果。
12. 把 Saga 补偿当成数据库回滚，没有处理补偿失败、不可逆动作和中间状态。


### 10.3 上线检查表

- [ ] 已定义 Thread、Run、Turn、Action 和 Event 的 ID 规则。
- [ ] 已定义模型停止、系统成功、系统失败和人工接管条件。
- [ ] 所有工具都有 Schema、超时、输出上限和错误分类。
- [ ] 高风险动作有 Policy、Approval、幂等键和审计记录。
- [ ] 执行环境限制文件、进程、网络、密钥和资源。
- [ ] Tool Result 先持久化，再进入下一轮模型调用。
- [ ] 多 Tool Call 已定义只读并行、依赖、失败传播和副作用顺序。
- [ ] 跨系统写入已定义 Saga、幂等、补偿、Pivot 和人工接管。
- [ ] 多 Tool Call 已定义只读并行、依赖、失败传播和副作用顺序。
- [ ] 跨系统写入已定义 Saga、幂等、补偿、Pivot 和人工接管。
- [ ] 多 Tool Call 已定义只读并行、依赖、失败传播和副作用顺序。
- [ ] 跨系统写入已定义 Saga、幂等、补偿、Pivot 和人工接管。
- [ ] 大结果保存为 Artifact，并向模型披露摘要和回读方式。
- [ ] Run 支持预算、取消、Checkpoint、恢复和并发控制。
- [ ] Verifier 独立执行可重复的验收标准。
- [ ] 失败可以按模型、工具、权限、基础设施和验收分类。


## 11 本章小结

读者现在应该能够解释以下结论：

1. Harness 是模型外部的运行和控制系统。
2. Agent Loop 是 Harness 的核心循环，Harness 还负责权限、状态、恢复、预算和验收。
3. Skill、CLI 和 MCP 向 Harness 提供方法或能力；Framework 帮助实现 Harness。
4. 完整任务按照上下文装配、模型决策、策略检查、工具执行、结果持久化、再次推理和独立验收运行。
5. 同一轮多个 Tool Call 没有依赖和事务语义；Harness 必须定义批次模式与失败传播。
6. 副作用动作需要 Policy、沙箱、审批、幂等和审计共同约束。
7. Checkpoint 恢复需要事件、工作区、执行回执和预算状态，单独保存消息不足以恢复任务。
8. 一次操作跨越多个独立提交资源时，需要本地事务、Outbox、Saga、补偿或人工门禁。
9. Harness 的建设强度由任务时长、副作用、失败成本和并发量决定。


## 12 与最终项目的关系

第十三章的产研协作助手会使用 Harness 承载需求分析、代码与文档读取、测试反馈和协作系统操作。Harness 将统一管理工具权限、项目状态、长任务恢复、Artifact、人工审批和结果验收。

本章没有提前实现产品功能。这里建立的运行时协议会在 Capstone 中直接复用。

## 13 下一步

下一章进入 Agent Frameworks。我们会使用统一场景比较 PydanticAI、LangChain/LangGraph、AutoGen 等实现方式，观察它们如何表达结构化输出、工具、状态、持久化和执行图，并判断框架替我们实现了哪些 Harness 职责。

## 参考文献

[1] OPENAI. Unrolling the Codex agent loop[EB/OL]. (2026-01-23)[2026-07-17]. https://openai.com/index/unrolling-the-codex-agent-loop/.

[2] OPENAI. Unlocking the Codex harness: how we built the App Server[EB/OL]. [2026-07-17]. https://openai.com/index/unlocking-the-codex-harness/.

[3] OPENAI. Running Codex safely at OpenAI[EB/OL]. [2026-07-17]. https://openai.com/index/running-codex-safely/.

[4] OPENAI. Harness engineering: leveraging Codex in an agent-first world[EB/OL]. [2026-07-17]. https://openai.com/index/harness-engineering/.

[5] ANTHROPIC. Building effective agents[EB/OL]. [2026-07-17]. https://www.anthropic.com/engineering/building-effective-agents.

[6] ANTHROPIC. Effective harnesses for long-running agents[EB/OL]. [2026-07-17]. https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents.

[7] ANTHROPIC. Harness design for long-running application development[EB/OL]. (2026-03-24)[2026-07-17]. https://www.anthropic.com/engineering/harness-design-long-running-apps.

[8] ANTHROPIC. Scaling Managed Agents: decoupling the brain from the hands[EB/OL]. (2026-04-08)[2026-07-17]. https://www.anthropic.com/engineering/managed-agents.

[9] YANG J, JIMENEZ C E, WETTIG A, et al. SWE-agent: Agent-computer interfaces enable automated software engineering[J/OL]. arXiv:2405.15793, 2024. https://arxiv.org/abs/2405.15793.

[10] GARCIA-MOLINA H, SALEM K. Sagas[C]//Proceedings of the 1987 ACM SIGMOD International Conference on Management of Data. New York: ACM, 1987: 249-259. DOI:10.1145/38713.38742.

[11] AMAZON WEB SERVICES. Saga orchestration pattern[EB/OL]. [2026-07-23]. https://docs.aws.amazon.com/prescriptive-guidance/latest/cloud-design-patterns/saga-orchestration.html.

[12] MICROSOFT. Compensating Transaction pattern[EB/OL]. (2026-04-20)[2026-07-23]. https://learn.microsoft.com/en-us/azure/architecture/patterns/compensating-transaction.
